In [1]:
# =====================================================================
# LAB 2: TRIAGEM DE CHAMADOS COM NAIVE BAYES
# =====================================================================
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
dados_treino = [
    ("Urgente: o servidor caiu e a aplicacao esta fora", "incidente_critico"),
    ("Erro grave no banco de dados e sistema parou", "incidente_critico"),
    ("Como faco para alterar minha senha do e-mail", "duvida_suporte"),
    ("Qual o procedimento para solicitar novo mouse", "duvida_suporte"),
    ("Gostaria de saber o horario de funcionamento", "fora_escopo")
]
stopwords = ["o", "a", "e", "do", "da", "para", "de"]
# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 2.1: Filtre as stopwords mantendo os tokens válidos
    tokens_filtrados = [t for t in tokens if t not in stopwords]

    return " ".join(tokens_filtrados)
X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]
# 2. VETORIZAÇÃO E TREINAMENTO (NAIVE BAYES)
vectorizer = TfidfVectorizer()
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)
# TODO 2.2: Instancie o modelo Naive Bayes (MultinomialNB)
modelo_nb = MultinomialNB()

In [2]:
# TODO 2.3: Treine o modelo com X_vetorizado e y_treino
modelo_nb.fit(X_vetorizado, y_treino)
# 3. EXTRAÇÃO DE ENTIDADE (CÓDIGO DE PROTOCOLO)
def extrair_protocolo(texto):
    match = re.search(r'INC-?\d{4}', texto, re.IGNORECASE)
    # TODO 2.4: Retorne a string do protocolo encontrado em maiúsculas ou None
    return match.group(0).upper() if match else None
# 4. PIPELINE NLU
def processar_triagem(mensagem_usuario, threshold=0.60):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo_nb.predict_proba(vetor_input)[0]

    maior_confianca = max(probas)
    intencao_prevista = modelo_nb.classes_[probas.argmax()]

    # TODO 2.5: Retorne dicionário de FALLBACK ou SUCESSO
    if maior_confianca < threshold:
        return {"status": "FALLBACK", "confianca": round(maior_confianca, 4), "mensagem": "Nao foi possivel classificar. Encaminhando para analise manual."}
    else:
        protocolo = extrair_protocolo(mensagem_usuario)
        return {"status": "SUCESSO", "intencao": intencao_prevista, "confianca": round(maior_confianca, 4), "protocolo": protocolo}


In [3]:
# --- TESTE 2 ---
if __name__ == "__main__":
    print("=== LAB 2 - NAIVE BAYES TRIAGEM ===")
    print()
    print("Teste 1:", processar_triagem("Urgente: servidor caiu no protocolo INC-9982"))
    print("Teste 2:", processar_triagem("Como faco para trocar minha senha"))
    print("Teste 3:", processar_triagem("Quero saber o horario de funcionamento"))
    print("Teste 4:", processar_triagem("Sistema fora do ar protocolo INC-4455"))
    print()
    print("--- Testes com threshold menor (0.45) ---")
    print("Teste 5:", processar_triagem("Urgente: servidor caiu no protocolo INC-9982", threshold=0.45))
    print("Teste 6:", processar_triagem("Como faco para trocar minha senha", threshold=0.45))
    print("Teste 7:", processar_triagem("Sistema fora do ar protocolo INC-4455", threshold=0.45))

=== LAB 2 - NAIVE BAYES TRIAGEM ===

Teste 1: {'status': 'FALLBACK', 'confianca': np.float64(0.5456), 'mensagem': 'Nao foi possivel classificar. Encaminhando para analise manual.'}
Teste 2: {'status': 'FALLBACK', 'confianca': np.float64(0.5583), 'mensagem': 'Nao foi possivel classificar. Encaminhando para analise manual.'}
Teste 3: {'status': 'FALLBACK', 'confianca': np.float64(0.3717), 'mensagem': 'Nao foi possivel classificar. Encaminhando para analise manual.'}
Teste 4: {'status': 'FALLBACK', 'confianca': np.float64(0.5009), 'mensagem': 'Nao foi possivel classificar. Encaminhando para analise manual.'}

--- Testes com threshold menor (0.45) ---
Teste 5: {'status': 'SUCESSO', 'intencao': np.str_('incidente_critico'), 'confianca': np.float64(0.5456), 'protocolo': 'INC-9982'}
Teste 6: {'status': 'SUCESSO', 'intencao': np.str_('duvida_suporte'), 'confianca': np.float64(0.5583), 'protocolo': None}
Teste 7: {'status': 'SUCESSO', 'intencao': np.str_('incidente_critico'), 'confianca': np.fl